# 📊 AIOS Quant Clustering & Anomaly Detection

Кластеризация 24 крипто-активов по рыночному поведению и детектирование аномальных объёмов (следы маркет-мейкеров / пампов).

**CPU достаточно** (можно T4 для автокодировщика).

In [ ]:
!pip install -q ccxt pandas numpy scikit-learn matplotlib
import ccxt, pandas as pd, numpy as np
print('✅ Зависимости установлены')

In [ ]:
# === ЯЧЕЙКА 2: Сбор данных по 24 активам ===
cl = ccxt.binance()
cl.load_markets()
symbols = ['BTC/USDT','ETH/USDT','BNB/USDT','SOL/USDT','XRP/USDT','ADA/USDT','DOGE/USDT','AVAX/USDT',
           'LINK/USDT','DOT/USDT','MATIC/USDT','LTC/USDT','TRX/USDT','ATOM/USDT','UNI/USDT','ETC/USDT',
           'FIL/USDT','APT/USDT','NEAR/USDT','ARB/USDT','OP/USDT','SUI/USDT','TIA/USDT','SEI/USDT']
data = {}
for s in symbols:
    try:
        o = cl.fetch_ohlcv(s, '1h', limit=500)
        data[s] = pd.DataFrame(o, columns=['ts','open','high','low','close','volume'])
    except Exception as e:
        print('skip', s, e)
print('✅ Собрано активов:', len(data))

In [ ]:
# === ЯЧЕЙКА 3: Кластеризация по поведению ===
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import numpy as np

feats = {}
for s, df in data.items():
    ret = df['close'].pct_change().dropna()
    vol = df['volume']
    feats[s] = [
        ret.mean()*100,           # средняя доходность %
        ret.std()*100,            # волатильность %
        vol.mean(),               # средний объём
        (vol.pct_change()>3).mean(),  # доля аномальных всплесков объёма
    ]
X = StandardScaler().fit_transform(np.array(list(feats.values())))
k = KMeans(n_clusters=4, random_state=42, n_init=10).fit(X)
for lbl in range(k.n_clusters):
    members = [s for i, s in enumerate(feats) if k.labels_[i]==lbl]
    print(f'Кластер {lbl}: {members}')

In [ ]:
# === ЯЧЕЙКА 4: Детекция аномалий (Isolation Forest) ===
from sklearn.ensemble import IsolationForest

iso = IsolationForest(contamination=0.05, random_state=42).fit(X)
preds = iso.predict(X)
anomalies = [s for i, s in enumerate(feats) if preds[i]==-1]
print('Аномальные активы:', anomalies)

# Аномальные всплески объёма внутри актива (маркет-мейкер/памп)
for s in data:
    vol = data[s]['volume']
    thresh = vol.rolling(24).mean() + 3*vol.rolling(24).std()
    spikes = (vol > thresh).sum()
    if spikes > 0:
        print(f'  {s}: {spikes} аномальных часов объёма')

In [ ]:
# === ЯЧЕЙКА 5: Корреляционная матрица ===
import pandas as pd
rets = pd.DataFrame({s: data[s]['close'].pct_change() for s in data}).dropna()
corr = rets.corr()
print('Размер матрицы корреляций:', corr.shape)
# топ-5 пар с самой высокой корреляцией
flat = corr.unstack().sort_values(ascending=False)
flat = flat[flat < 1.0].drop_duplicates()
print(flat.head(5))
corr.to_csv('models/asset_correlations.csv')
print('✅ Сохранено: models/asset_correlations.csv')